In [6]:
import pandas as pd
import networkx as nx

def generar_grafo(ruta_csv, nombre_universo):
    # 1. Leer el CSV limpio
    df = pd.read_csv(ruta_csv)
    
    # 2. Crear el objeto Grafo en NetworkX
    G = nx.from_pandas_edgelist(
        df, 
        source='source', 
        target='target', 
        edge_attr='weight', 
        create_using=nx.Graph()
    )
    G.remove_edges_from(nx.selfloop_edges(G))  # Eliminar autobucles si los hubiera

    # 3. Calcular centralidades
    # Invertimos el peso para Betweenness: a más interacciones, menor distancia
    for u, v, d in G.edges(data=True):
        d['distance'] = 1.0 / d['weight'] if d['weight'] > 0 else 1.0

    weighted_degree = dict(G.degree(weight='weight'))
    betweenness = nx.betweenness_centrality(G, weight='distance', normalized=True)
    eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)

    # 4. Detectar comunidades (Algoritmo de Louvain)
    comunidades = nx.community.louvain_communities(G, weight='weight', seed=42)
    mapa_comunidades = {
        nodo: idx for idx, com in enumerate(comunidades) for nodo in com
    }

    # 5. Guardar todos los cálculos dentro de los nodos del grafo
    nx.set_node_attributes(G, weighted_degree, 'weighted_degree')
    nx.set_node_attributes(G, betweenness, 'betweenness')
    nx.set_node_attributes(G, eigenvector, 'eigenvector')
    nx.set_node_attributes(G, mapa_comunidades, 'comunidad')

    # 6. Exportar el grafo a formato .gexf para Gephi
    ruta_gexf = f"../data/processed/gephi_exports/{nombre_universo}.gexf"
    nx.write_gexf(G, ruta_gexf)
    
    print(f"✅ Grafo '{nombre_universo}' generado: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas -> Guardado en {ruta_gexf}")
    return G

# Ejecutar para tus 3 universos (ajusta el nombre de Marvel si lo guardaste distinto)
G_got = generar_grafo("../data/processed/got/got_full_interactions_edges_clean.csv", "got_global")
G_sw = generar_grafo("../data/processed/starwars/starwars_full_interactions_edges_clean.csv", "starwars_global")
G_marvel = generar_grafo("../data/processed/marvel/marvel_edges_clean.csv", "marvel_global")